# GPT Fine-Tuning Basics (tiny causal LM)

Implements a tiny causal transformer, trains on a small corpus, and samples text.

_Last rebuild: **2026-02-16 03:28:46**_

In [1]:
import numpy as np
import torch
import torch.nn as nn

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


In [2]:
text = '''
In industrial ML, reproducibility beats vibes.
We log configs, seed everything, and validate outputs.
Fine-tuning is just gradient descent on a pretrained prior.
'''.strip()

chars=sorted(set(text))
stoi={c:i for i,c in enumerate(chars)}
itos={i:c for c,i in stoi.items()}
enc=torch.tensor([stoi[c] for c in text], dtype=torch.long)

vocab=len(chars)
print('vocab', vocab, 'len', len(enc))

vocab 30 len 161


In [3]:
block=64
X=[]; y=[]
for i in range(len(enc)-block-1):
    X.append(enc[i:i+block]); y.append(enc[i+1:i+block+1])
X=torch.stack(X); y=torch.stack(y)
split=int(0.9*len(X))
Xtr,Xte=X[:split],X[split:]; ytr,yte=y[:split],y[split:]
Xtr.shape

torch.Size([86, 64])

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab, d=64, heads=4, layers=2, block=64):
        super().__init__()
        self.tok=nn.Embedding(vocab,d)
        self.pos=nn.Embedding(block,d)
        enc_layer=nn.TransformerEncoderLayer(d_model=d, nhead=heads, batch_first=True)
        self.tr=nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.lm=nn.Linear(d, vocab)
        self.block=block
    def forward(self, idx):
        B,T=idx.shape
        pos=torch.arange(T, device=idx.device)
        x=self.tok(idx)+self.pos(pos)[None,:,:]
        mask=torch.triu(torch.ones(T,T,device=idx.device), diagonal=1).bool()
        x=self.tr(x, mask=mask)
        return self.lm(x)

model=TinyGPT(vocab, block=block).to(device)
opt=torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn=nn.CrossEntropyLoss()

def batch(bs=64):
    ix=torch.randint(0, Xtr.shape[0], (bs,))
    return Xtr[ix].to(device), ytr[ix].to(device)

for step in range(250):
    xb,yb=batch()
    logits=model(xb)
    loss=loss_fn(logits.view(-1, vocab), yb.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1)%50==0:
        print('step', step+1, 'loss', float(loss))


step 50 loss 1.9028071165084839
step 100 loss 1.424227237701416
step 150 loss 0.8990617394447327
step 200 loss 0.49935778975486755
step 250 loss 0.2982509434223175


In [5]:
@torch.no_grad()
def sample(prefix='Fine-tuning ', n=180, temp=0.9):
    model.eval()
    ctx=torch.tensor([stoi.get(c,0) for c in prefix], dtype=torch.long, device=device)[None,:]
    for _ in range(n):
        inp=ctx[:,-block:]
        logits=model(inp)[:,-1,:]/temp
        probs=torch.softmax(logits, dim=-1)
        nxt=torch.multinomial(probs, 1)
        ctx=torch.cat([ctx,nxt], dim=1)
    return ''.join(itos[int(i)] for i in ctx[0].cpu())

print(sample())

Fine-tuning g andede-tuntus inis is justra just digrust dient ent gradesescent varad vipralidid outs.
Fine-tuning inist just gradie dgradievateratput de-tunint ieng jut gra diestcent adent ont


In [6]:
print('DONE 2026-02-16 03:28:46')

DONE 2026-02-16 03:28:46
